# RSA Lab: Classical Mathematical Implementation 


Implement and analyse RSA using basic primality testing, private computation of Euler's totient, Euclidean and Extended Euclidean algorithms, integer encryption and decryption, and an explicit totient-leak attack.

The notebook explains the correct classical procedure, not a complete secure protocol.

## 1. Public-key cryptography and RSA

RSA uses two related keys:

- **public key** $(n,e)$: shared openly;
- **private key** $(n,d)$: retained by the owner.

The owner secretly selects distinct primes $p,q$, calculates $n=pq$, privately constructs $d$, and publishes only $(n,e)$. A sender encrypts an integer $m$ using the public key:

$$c=m^e\bmod n.$$

The owner decrypts it using the private key:

$$m=c^d\bmod n.$$

The values $p,q,\phi(n)$ and $d$ must not be disclosed.

## 2. Basic primality testing

If $N=ab$ is composite, at least one factor is at most $\sqrt N$; otherwise $a>\sqrt N$ and $b>\sqrt N$ would imply $ab>N$. Hence it is enough to test divisors through $\sqrt N$.

### Algorithm `IsPrime(N)`

1. Reject $N<2$.
2. Accept $N=2$.
3. Reject even $N>2$.
4. Test $3,5,7,\ldots$ while $d^2\le N$.
5. Reject if a divisor is found; otherwise accept.

Worst case: $O(\sqrt N)$ divisions. If $N$ has $b$ bits, $N\approx2^b$, so this is $O(2^{b/2})$. It is appropriate for small transparent examples, not large prime generation.

In [1]:
def is_prime(n):
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    divisor = 3
    while divisor * divisor <= n:
        if n % divisor == 0:
            return False
        divisor += 2
    return True

for value in [2, 29, 51, 53, 61, 91]:
    print(value, is_prime(value))

2 True
29 True
51 False
53 True
61 True
91 False


## 3. Why the Euclidean algorithm is required

RSA must choose $e$ for which a modular inverse modulo $\phi(n)$ exists. The inverse exists exactly when

$$\gcd(e,\phi(n))=1.$$

Thus the Euclidean algorithm is required to **validate the proposed public exponent**. If the gcd is not 1, no private exponent $d$ can satisfy

$$ed\equiv1\pmod{\phi(n)},$$

so that $e$ must be rejected.

For $e=17$ and $\phi(n)=3120$:

$$
\begin{aligned}
3120&=183(17)+9,\\
17&=1(9)+8,\\
9&=1(8)+1,\\
8&=8(1)+0.
\end{aligned}
$$

The last nonzero remainder is 1. Therefore $\gcd(17,3120)=1$, and an inverse exists.

## 4. Why the Extended Euclidean algorithm is required

The ordinary Euclidean algorithm only confirms that the gcd is 1. RSA still needs the actual inverse $d$.

The Extended Euclidean algorithm finds $x,y$ such that

$$ex+\phi(n)y=1.$$

Working backward in the example:

$$
\begin{aligned}
1&=9-8\\
 &=9-(17-9)\\
 &=2(9)-17\\
 &=2(3120-183\cdot17)-17\\
 &=2(3120)-367(17).
\end{aligned}
$$

Hence

$$(-367)(17)+2(3120)=1.$$

Modulo 3120,

$$(-367)(17)\equiv1\pmod{3120}.$$

Therefore

$$d\equiv-367\equiv2753\pmod{3120}.$$

In short:

- **Euclidean algorithm:** checks whether $e$ is valid.
- **Extended Euclidean algorithm:** efficiently computes $d$.

A trial search for $d$ could be enormous. Extended Euclid requires only $O(\log\phi(n))$ arithmetic iterations.

In [2]:
def gcd(a, b):
    while b != 0:
        a, b = b, a % b
    return a


def extended_gcd(a, b):
    old_r, r = a, b
    old_x, x = 1, 0
    old_y, y = 0, 1
    while r != 0:
        q = old_r // r
        old_r, r = r, old_r - q * r
        old_x, x = x, old_x - q * x
        old_y, y = y, old_y - q * y
    return old_r, old_x, old_y


def modular_inverse(a, modulus):
    g, x, unused_y = extended_gcd(a, modulus)
    if g != 1:
        raise ValueError("Modular inverse does not exist.")
    return x % modulus

## 5. Explicit RSA key-generation procedure

**Input:** Distinct primes $p,q$ and proposed exponent $e$.  
**Output:** Public key $(n,e)$ and private key $(n,d)$.

1. Verify that $p,q$ are prime and distinct.
2. Compute $n=pq$.
3. Privately compute
   $$\phi(n)=(p-1)(q-1).$$
4. Use Euclid to verify
   $$1<e<\phi(n),\qquad\gcd(e,\phi(n))=1.$$
5. Use Extended Euclid to compute
   $$d=e^{-1}\pmod{\phi(n)}.$$
6. Publish only $(n,e)$.
7. Keep $p,q,\phi(n),d$ secret.

Computing $\phi(n)$ privately is normal because the owner knows $p,q$. Disclosing it is not normal and breaks the key.

In [3]:
def generate_rsa_keys(p, q, e):
    if not is_prime(p) or not is_prime(q):
        raise ValueError("p and q must be prime.")
    if p == q:
        raise ValueError("p and q must be distinct.")
    n = p * q
    phi = (p - 1) * (q - 1)  # private temporary value
    if not (1 < e < phi) or gcd(e, phi) != 1:
        raise ValueError("e must satisfy 1 < e < phi(n) and gcd(e,phi(n))=1.")
    d = modular_inverse(e, phi)
    return (n, e), (n, d)

## 6. Encryption and decryption

Represent the message by an integer satisfying $0\le m<n$.

- Encryption with public key: $c=m^e\bmod n$.
- Decryption with private key: $m'=c^d\bmod n$.

The sender does not need $p,q$ or $\phi(n)$. The three-argument `pow` performs efficient modular exponentiation without constructing the full power.

In [4]:
def encrypt(message, public_key):
    n, e = public_key
    if not (0 <= message < n):
        raise ValueError("Message must satisfy 0 <= m < n.")
    return pow(message, e, n)


def decrypt(ciphertext, private_key):
    n, d = private_key
    if not (0 <= ciphertext < n):
        raise ValueError("Ciphertext must satisfy 0 <= c < n.")
    return pow(ciphertext, d, n)

## 7. Complete example

Choose $p=61$, $q=53$, $e=17$. Then

$$n=61\cdot53=3233,\qquad \phi(n)=60\cdot52=3120.$$

As $\gcd(17,3120)=1$,

$$d=17^{-1}\pmod{3120}=2753,$$

because $17\cdot2753=15(3120)+1$.

Therefore

$$K_{\rm public}=(3233,17),\qquad K_{\rm private}=(3233,2753).$$

For $m=65$:

$$c=65^{17}\bmod3233=2790,$$

$$2790^{2753}\bmod3233=65.$$

In [5]:
public_key, private_key = generate_rsa_keys(61, 53, 17)
message = 65
ciphertext = encrypt(message, public_key)
recovered = decrypt(ciphertext, private_key)
print("Public key:", public_key)
print("Private key:", private_key)
print("Message:", message)
print("Ciphertext:", ciphertext)
print("Recovered:", recovered)

Public key: (3233, 17)
Private key: (3233, 2753)
Message: 65
Ciphertext: 2790
Recovered: 65


## 8. Why decryption works

Since $ed\equiv1\pmod{\phi(n)}$, there is a $k$ such that $ed=1+k\phi(n)$.

If $\gcd(m,n)=1$, Euler's theorem gives $m^{\phi(n)}\equiv1\pmod n$. Hence

$$m^{ed}=m^{1+k\phi(n)}=m(m^{\phi(n)})^k\equiv m\pmod n.$$

If $m$ shares a factor with $n$, work modulo $p$ and $q$ separately. If $p\mid m$, both $m^{ed}$ and $m$ are 0 modulo $p$; otherwise Fermat's theorem applies. Thus $m^{ed}\equiv m\pmod p$, and similarly modulo $q$. Since $p,q$ are distinct,

$$m^{ed}\equiv m\pmod{pq}=m\pmod n.$$

Therefore $c^d\equiv(m^e)^d\equiv m\pmod n$.

## 9. Disclosure of $\phi(n)$ breaks RSA

If an attacker knows $n,e$ and $\phi(n)$, then

$$\phi(n)=(p-1)(q-1)=n-(p+q)+1,$$

so

$$S=p+q=n-\phi(n)+1.$$

The secret primes are roots of

$$x^2-Sx+n=0,$$

hence

$$p,q=\frac{S\pm\sqrt{S^2-4n}}2.$$

The attacker also computes $d=e^{-1}\pmod{\phi(n)}$.

For $n=3233$, $e=17$, leaked $\phi(n)=3120$:

$$S=114,\quad S^2-4n=64,$$

so $p,q=(114\pm8)/2=61,53$, and $d=2753$. The key is fully broken.

In [6]:
def integer_square_root(n):
    low, high = 0, n
    while low <= high:
        mid = (low + high) // 2
        if mid * mid <= n:
            low = mid + 1
        else:
            high = mid - 1
    return high


def break_rsa_with_phi(n, e, leaked_phi, ciphertext):
    S = n - leaked_phi + 1
    delta = S * S - 4 * n
    root = integer_square_root(delta)
    if delta < 0 or root * root != delta or (S + root) % 2 != 0:
        raise ValueError("Inconsistent phi(n).")
    p, q = (S + root) // 2, (S - root) // 2
    d = modular_inverse(e, leaked_phi)
    return p, q, d, pow(ciphertext, d, n)

p_found, q_found, d_found, message_found = break_rsa_with_phi(3233,17,3120,2790)
print(p_found, q_found, d_found, message_found)

61 53 2753 65


## 10. What RSA is useful for

### A. Protecting or transporting a short secret

RSA is well suited to protecting a short value, especially a randomly generated symmetric session key. A symmetric algorithm then handles the large message. This is hybrid encryption. RSA-based integer-factorization schemes are specified for key establishment and key transport.

### B. Digital signatures

RSA can support origin authentication, integrity and public verification. The private key signs and the public key verifies. A complete signature scheme uses standardized message encoding rather than raw textbook RSA.
### C. Certificates and public-key infrastructure

An RSA public key may be placed in a certificate that binds an identity to that key. It can then support certificate-based authentication and signature verification.

## 11. What RSA is not advisable for

### A. Direct encryption of large data

RSA is slower than symmetric encryption and operates only on messages smaller than $n$. It should not encrypt a file, image, video or database block by block. Use RSA for a short key and symmetric encryption for bulk data. RSA key transport is a recognized use.
### B. Raw RSA for actual confidentiality

Textbook RSA is deterministic: the same $m$ and public key always give the same $c$. Its algebraic structure is also manipulable. The raw operation in this notebook is correct mathematics but not a complete secure encryption scheme. Complete RSA schemes include standardized encoding and padding. 

### C. Password storage

RSA encryption is not password hashing. Password storage requires a dedicated salted, deliberately slow password-hashing or password-based key-derivation method.

### D. Integrity by encryption alone

Encryption alone does not establish who sent the data or guarantee that it was unchanged. Use an authenticated-encryption mechanism, message-authentication mechanism, or proper signature scheme as required.

### E. Highly constrained environments

RSA keys and signatures are comparatively large, and private operations are expensive. Another approved public-key family may be preferable where bandwidth, memory or energy is severely constrained.

### F. Post-quantum protection

RSA depends on the classical difficulty of integer factorization and is not post-quantum secure. Long-lived systems requiring protection against future cryptographically relevant quantum computers should plan migration to approved post-quantum mechanisms.

This lab therefore implements the correct classical mathematics, not a deployable protocol. A complete system also needs standardized encoding, secure randomness, key storage, side-channel resistance and key management. 

## 12. Running-time analysis

- Trial primality test: $O(\sqrt p)$ divisions, exponential in the bit length.
- Euclidean algorithm: $O(\log\phi(n))$ remainder steps.
- Extended Euclidean algorithm: $O(\log\phi(n))$ arithmetic iterations.
- Encryption: $O(\log e)$ modular multiplications.
- Decryption: $O(\log d)$ modular multiplications.
- Totient-leak attack: an integer square root plus a modular inverse, efficient compared with factoring an unknown large $n$.



## 13. Complete procedure at a glance

1. Secretly choose distinct primes $p,q$.
2. Compute $n=pq$ and privately compute $\phi(n)=(p-1)(q-1)$.
3. Use Euclid to verify $\gcd(e,\phi(n))=1$.
4. Use Extended Euclid to find $d=e^{-1}\pmod{\phi(n)}$.
5. Publish $(n,e)$; keep $p,q,\phi(n),d$ secret.
6. Encrypt $m<n$ by $c=m^e\bmod n$.
7. Decrypt by $m=c^d\bmod n$.
8. If $\phi(n)$ leaks, recover $p,q,d$ and break RSA.
9. Use textbook RSA for mathematical study; use standardized schemes for actual applications.

## 14. Exercises

### Exercise 1: Construct RSA

Use $p=47$, $q=59$, $e=17$, $m=123$.

1. Verify primality.
2. Compute $n$ and privately compute $\phi(n)$.
3. Show Euclidean steps proving $\gcd(e,\phi(n))=1$.
4. Use Extended Euclid to calculate $d$.
5. Encrypt and decrypt $m$.
6. Identify public and private quantities.

### Exercise 2: Break RSA after a totient leak

Given $(n,e)=(3233,17)$, leaked $\phi(n)=3120$, and $c=2790$:

1. compute $S=n-\phi(n)+1$;
2. solve $x^2-Sx+n=0$;
3. recover $p,q,d$;
4. decrypt $c$;
5. explain why $\phi(n)$ must never be disclosed.

In [22]:
#Exercise 1
## FUNCTIONS:
def is_prime(n):
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    divisor = 3
    while divisor * divisor <= n:
        if n % divisor == 0:
            return False
        divisor += 2
    return True

for value in [2, 29, 51, 53, 61, 91]:
    print(value, is_prime(value))

def gcd(a, b):
    while b != 0:
        a, b = b, a % b
    return a

def extended_gcd(a, b):
    old_r, r = a, b
    old_x, x = 1, 0
    old_y, y = 0, 1
    while r != 0:
        q = old_r // r
        old_r, r = r, old_r - q * r
        old_x, x = x, old_x - q * x
        old_y, y = y, old_y - q * y
    return old_r, old_x, old_y


def modular_inverse(a, modulus):
    g, x, unused_y = extended_gcd(a, modulus)
    if g != 1:
        raise ValueError("Modular inverse does not exist.")
    return x % modulus

def encrypt(message, public_key):
    n, e = public_key
    if not (0 <= message < n):
        raise ValueError("Message must satisfy 0 <= m < n.")
    return pow(message, e, n)


def decrypt(ciphertext, private_key):
    n, d = private_key
    if not (0 <= ciphertext < n):
        raise ValueError("Ciphertext must satisfy 0 <= c < n.")
    return pow(ciphertext, d, n)


## MAIN
p=47
q=59
e=17
m=123

if not is_prime(p) or not is_prime(q):
    raise ValueError("p and q must be prime.")

if p == q:
    raise ValueError("p and q must be distinct.")
    
n = p * q
phi = (p - 1) * (q - 1)

def euclidean_gcd(a, b):
    print("\n--- Euclidean Algorithm Steps ---")
    steps = []
    while b != 0:
        q_val = a // b
        r = a % b
        steps.append(f"{a} = {q_val} * {b} + {r}")
        a, b = b, r
    for step in steps:
        print(step)
    return a

gcd_val = euclidean_gcd(e, phi)
print(f"gcd(e, phi(n)) = {gcd_val}")

d = modular_inverse(e, phi)

cipher=encrypt(m,(n,e))
message=decrypt(cipher,(n,d))

print(f"\nEncrypted: {cipher}, Decrypted: {message}")

print(f"\nPublic Key: (e = {e}, n = {n})")
print(f"Private Key: (d = {d}, n = {n}) \n[Secret: p={p}, q={q}, phi(n)={phi}]")


--- Euclidean Algorithm Steps ---
17 = 0 * 2668 + 17
2668 = 156 * 17 + 16
17 = 1 * 16 + 1
16 = 16 * 1 + 0
gcd(e, phi(n)) = 1

Encrypted: 1924, Decrypted: 123

Public Key: (e = 17, n = 2773)
Private Key: (d = 157, n = 2773) 
[Secret: p=47, q=59, phi(n)=2668]


In [21]:
## FUNCTIONS:
def extended_gcd(a: int, b: int) -> tuple[int, int, int]:
    """Extended Euclidean Algorithm to find gcd and bezout coefficients."""
    if a == 0:
        return b, 0, 1
    gcd, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return gcd, x, y


def modular_inverse(e: int, phi: int) -> int:
    """Calculates the modular inverse of e modulo phi."""
    gcd, x, _ = extended_gcd(e, phi)
    if gcd != 1:
        raise ValueError("Modular inverse does not exist (gcd != 1).")
    return x % phi


def integer_square_root(n: int) -> int:
    """Computes the integer square root using binary search."""
    low, high = 0, n
    while low <= high:
        mid = (low + high) // 2
        if mid * mid <= n:
            low = mid + 1
        else:
            high = mid - 1
    return high


def break_rsa_with_phi(n: int, e: int, leaked_phi: int, ciphertext: int):
    # 1. Compute S = n - phi(n) + 1
    S = n - leaked_phi + 1
    print(f"1. S = n - phi(n) + 1 = {n} - {leaked_phi} + 1 = {S}")

    # 2. Solve x^2 - Sx + n = 0 using the quadratic formula / square root of discriminant
    delta = S * S - 4 * n
    root = integer_square_root(delta)
    print(f"2. Discriminant delta = S^2 - 4n = {delta}")
    print(f"   Quadratic equation: x^2 - {S}x + {n} = 0")

    if delta < 0 or root * root != delta or (S + root) % 2 != 0:
        raise ValueError("Inconsistent phi(n).")

    # 3. Recover p, q, and d
    p = (S + root) // 2
    q = (S - root) // 2
    d = modular_inverse(e, leaked_phi)
    print(f"3. Recovered primes: p = {p}, q = {q}")
    print(f"   Recovered private exponent: d = {d}")

    # 4. Decrypt c
    plaintext = pow(ciphertext, d, n)
    print(f"4. Decrypted plaintext for c = {ciphertext} is m = {plaintext}")

    return p, q, d, plaintext

## MAIN
n_val = 3233
e_val = 17
phi_val = 3120
c_val = 2790

break_rsa_with_phi(n_val, e_val, phi_val, c_val)

1. S = n - phi(n) + 1 = 3233 - 3120 + 1 = 114
2. Discriminant delta = S^2 - 4n = 64
   Quadratic equation: x^2 - 114x + 3233 = 0
3. Recovered primes: p = 61, q = 53
   Recovered private exponent: d = 2753
4. Decrypted plaintext for c = 2790 is m = 65


(61, 53, 2753, 65)

#### EXPLANATION: 

Exposing $\phi(n)$ allows anyone to instantly calculate the sum and product of the secret primes $p$ and $q$, factoring $n$ and completely destroying the private key's security.